# Run the baselines on Kaggle (GPU T4, Gemma-2-2B-IT)

Designed for **Save & Run All**: no prompts, every phase resumable, a wall-clock budget below Kaggle's 12-hour limit, and results saved as the notebook's output.

**Notebook settings (once)**
- *Accelerator*: **GPU T4 x2** (one GPU is used). *Internet*: **on** (clone, pip, Hugging Face).
- *Add-ons → Secrets*: add `HF_TOKEN` (an HF read token whose account accepted the Gemma-2 licence) and attach it to this notebook. It is read into the process environment only and never printed or written.

**Each session**
1. To continue earlier work, attach the previous session's output (*Add Input → Your Work → this notebook's latest version*), or a dataset made from its `results.zip`. The restore cell copies the most complete `results/` it finds into `/kaggle/working/results`. With nothing attached, the run starts from scratch.
2. *Save Version → Save & Run All (Commit)*.
3. The output (`/kaggle/working`) holds `results/` (the resume state), `results.zip` and `session_status.json`. Finished stages are skipped next time; a stage cut off by the deadline resumes where it stopped.

The whole run needs roughly 20-35 T4-hours, so expect 2-3 sessions (Kaggle's weekly GPU quota is about 30 h). See `docs/DECISIONS.md` D15/D16.

In [ ]:
# 1. Session clock, paths, environment. Nothing here needs the repo.
import os, sys, time, json, subprocess
T0 = time.time()
SESSION_HOURS = 11.5   # Kaggle stops GPU sessions at 12 h
RESERVE_MIN = 25       # kept free at the end for zipping the results
RESULTS = '/kaggle/working/results'
REPO = '/tmp/Reasoning-Failure-Diagnosis'   # outside /kaggle/working: the code is not output
TARGETS = ['phase0', 'baseline-a', 'traces', 'baseline-b', 'pilot', 'pilot-sample', 'report']
os.makedirs(RESULTS, exist_ok=True)
os.environ.update(
    RFD_RESULTS_DIR=RESULTS,
    RFD_SCRATCH_DIR='/tmp/rfd_scratch',          # assembled activation arrays (not output)
    PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True',
    CUDA_VISIBLE_DEVICES='0',
    TOKENIZERS_PARALLELISM='false',
)
DEADLINE = T0 + SESSION_HOURS * 3600 - RESERVE_MIN * 60

def sh(cmd):
    """Run a shell command, show its output in the cell, raise on failure."""
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end='')
    if p.wait():
        raise RuntimeError(f'command failed ({p.returncode}): {cmd}')

print('results ->', RESULTS, '| phases must finish by', time.strftime('%H:%M', time.localtime(DEADLINE)))

In [ ]:
# 2. HF token from Kaggle secrets -> environment (never printed).
from kaggle_secrets import UserSecretsClient
try:
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    raise RuntimeError('No HF_TOKEN secret: Add-ons -> Secrets -> add HF_TOKEN and attach it') from None
print('HF_TOKEN set:', bool(os.environ.get('HF_TOKEN')))

In [ ]:
# 3. Get or update the code.
if os.path.isdir(REPO):
    sh(f'git -C {REPO} pull --ff-only')
else:
    sh(f'git clone --depth 1 https://github.com/mehul24d/Reasoning-Failure-Diagnosis.git {REPO}')
os.chdir(REPO)
COMMIT = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip()
sh('git log --oneline -1')

In [ ]:
# 4. Install (scripts/install_gpu_env.py; DECISIONS D16/D17): Kaggle's torch /
#    torchvision / torchaudio / triton / nvidia-* are frozen into a constraints file
#    (pip list --format=freeze), so the preinstalled CUDA torch is never replaced;
#    wheels only, so nothing is compiled; results-relevant libraries at their pinned
#    versions, Kaggle's own numpy / scipy / pandas / matplotlib kept unless a pin
#    needs otherwise. Full pip output is shown; versions -> install_report.json.
import importlib.metadata as md
torch_before = md.version('torch')
sh(f'{sys.executable} scripts/install_gpu_env.py --report /kaggle/working/install_report.json')
sh(f'{sys.executable} -m pip install -q -e . --no-deps')
import torch
assert torch.cuda.is_available(), 'CUDA is not available: set Accelerator = GPU T4 x2'
assert md.version('torch') == torch_before, f'torch changed: {torch_before} -> {md.version("torch")}'
print('torch', torch.__version__, '| CUDA', torch.version.cuda, '|', torch.cuda.get_device_name(0))

In [ ]:
# 5. Restore: copy the most complete results/ from any attached input (a previous
#    version's output, or a dataset made from results.zip). Existing files are never
#    overwritten, so re-running is harmless; with no input this is a fresh start.
sys.path.insert(0, f'{REPO}/src')   # the editable install is only seen by new processes
from rfd.utils.session import restore_results
print(json.dumps(restore_results('/kaggle/input', RESULTS), indent=2))

In [ ]:
# 6. Smoke test (a few minutes; raises, i.e. stops the run, on any FAIL): unit tests,
#    Gemma access, pure-greedy generation config, one BOS, system text in the first
#    user turn, worst-case batches fit the T4, activations fit float16, and a
#    zero-strength steering edit changes nothing.
from huggingface_hub import hf_hub_download
hf_hub_download('google/gemma-2-2b-it', 'config.json')
sh('make test-fast')
sh('make smoke')

In [ ]:
# 7. All phases in order, until the session deadline. A phase still running at the
#    deadline is stopped and resumes in the next session; a failing phase stops the
#    chain (later phases depend on it). This cell never raises, so step 8 always runs.
from rfd.utils.session import run_targets
try:
    STATUS = run_targets(TARGETS, DEADLINE)
except Exception as e:  # noqa: BLE001
    STATUS = [{'target': '?', 'status': f'session error: {e!r}'}]
for row in STATUS:
    print(f"{row['target']:>13}  {row['status']}")

In [ ]:
# 8. End of session: zip results/ and record what happened (also appended to
#    results/sessions.jsonl, which travels with the results).
from rfd.utils.session import zip_results, write_status
z = zip_results(RESULTS, '/kaggle/working/results.zip')
session = {
    'finished': time.strftime('%Y-%m-%d %H:%M:%S'),
    'hours': round((time.time() - T0) / 3600, 2),
    'commit': COMMIT,
    'gpu': torch.cuda.get_device_name(0),
    'phases': STATUS,
    'zip': z,
    'all_complete': all(r['status'] == 'complete' for r in STATUS),
}
write_status('/kaggle/working/session_status.json', session)
with open(f'{RESULTS}/sessions.jsonl', 'a') as f:
    f.write(json.dumps(session, default=str) + '\n')
print(json.dumps(session, indent=2, default=str))
print('\nAll phases complete.' if session['all_complete'] else
      '\nNot finished: attach this version\'s output as input and Save & Run All again.')